<a href="https://colab.research.google.com/github/Susan-sapkota-98/Nepali-asr-evaluation/blob/main/03_wer_cer_labels.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q jiwer

In [2]:
from pathlib import Path

import jiwer
import pandas as pd
from google.colab import drive

drive.mount("/content/drive")

samples_folder = Path("/content/drive/MyDrive/nepali-asr-evaluation/samples_20")
samples = pd.read_csv(samples_folder / "predictions_mms.csv", dtype=str, keep_default_na=False)
print(len(samples), "rows")

Mounted at /content/drive
20 rows


## WER and CER for each row

In [3]:
all_wer = []
all_cer = []
for i in range(len(samples)):
    correct = samples.loc[i, "text"]
    predicted = samples.loc[i, "mms_prediction"]
    all_wer.append(jiwer.wer(correct, predicted))
    all_cer.append(jiwer.cer(correct, predicted))

samples["wer"] = all_wer
samples["cer"] = all_cer

## Score for all 20 files

In [4]:
overall_wer = jiwer.wer(list(samples["text"]), list(samples["mms_prediction"]))
overall_cer = jiwer.cer(list(samples["text"]), list(samples["mms_prediction"]))
print("WER:", round(overall_wer * 100, 1), "%")
print("CER:", round(overall_cer * 100, 1), "%")

# the 5 worst rows
samples.sort_values("cer", ascending=False)[["text", "mms_prediction", "wer", "cer"]].head(5)

WER: 41.4 %
CER: 20.3 %


,text,mms_prediction,wer,cer
2,एमएस्सीका लागि,kanananananana aeaeaeaeaeanananr,1.000000,2.214286
15,२०३२ सालदेखि लोक,205 साल देखि लोक,1.000000,0.312500
12,जापान खाद्यान्नको मामलामा,जापान खातदेनेको माबलामा,0.666667,0.280000
10,उनी अन्य कृयाकलापहरूमा,उनी अन्यक्रिया कलाबहरूमा,0.666667,0.272727
19,त्यसरी उनले पुस्तकहरूको,यसरि उनले पुस्तकहरूप हो,1.000000,0.260870


In [5]:
# rows 10 to 19
for i in range(10, 20):
    print(i, samples.loc[i, "text"], "->", samples.loc[i, "mms_prediction"])

10 उनी अन्य कृयाकलापहरूमा -> उनी अन्यक्रिया कलाबहरूमा
11 आधार मानेर भएर -> आधरमानेर भएर
12 जापान खाद्यान्नको मामलामा -> जापान खातदेनेको माबलामा
13 बाँचे जसको मृत्यु -> बाँचे जसको मृर्त्यो
14 गीतहरूको सिर्जना गर्ने -> गीतहरूको सिर्जना गर्ने
15 २०३२ सालदेखि लोक -> 205 साल देखि लोक
16 यसले छाउराहरूलाई आफ्नो -> यसले छाउराहरूलाई आफ्नो
17 काम गर्न पनि -> काम गर्न पनि
18 लेख्ने विचार गर्नुभएको -> लेख्नी विचार गर्नुभएको
19 त्यसरी उनले पुस्तकहरूको -> यसरि उनले पुस्तकहरूप हो


In [7]:
my_labels = {
    0: "exact", 1: "exact", 2: "failed", 3: "exact", 4: "spelling",
    5: "spelling", 6: "real_mistake", 7: "spelling", 8: "spelling", 9: "real_mistake",
    10: "real_mistake", 11: "spacing", 12: "real_mistake", 13: "spelling", 14: "exact",
    15: "number", 16: "exact", 17: "exact", 18: "real_mistake", 19: "real_mistake",
}

samples["my_label"] = [my_labels[i] for i in range(len(samples))]
print("rows without label:", (samples["my_label"] == "").sum())
print(samples["my_label"].value_counts())

rows without label: 0
my_label
exact           6
real_mistake    6
spelling        5
failed          1
spacing         1
number          1
Name: count, dtype: int64


In [8]:
samples.groupby("my_label")[["wer","cer"]].mean().round(2)

,wer,cer
my_label,,
exact,0.00,0.00
failed,1.00,2.21
number,1.00,0.31
real_mistake,0.57,0.20
spacing,0.67,0.14
spelling,0.48,0.12


In [9]:
samples.to_csv(samples_folder / "labeled_20.csv", index=False)
print("saved")

saved


## Notes
- overall WER and CER:
- which label has the most rows:
- do "spelling" rows have a high WER?